# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="../assets/LC_Memory_before.png" style="width:500px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.3     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.5     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-community    | >=0.4.2  | 0.4.2     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/pyth

In [2]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///db/Chinook.db")

/var/folders/5d/nbt0zlvs563699mx1z5mslxm0000gn/T/ipykernel_39351/2306864394.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [7]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db)
):
    step["messages"][-1].pretty_print()  # print the last message in the stream
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_2JfzK3Th2TZJHcSejPAYZpIX)
 Call ID: call_2JfzK3Th2TZJHcSejPAYZpIX
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' AND (lower(name) LIKE '%invoice%' OR lower(name) LIKE '%customer%' OR lower(name) LIKE '%client%' OR lower(name) LIKE '%user%') ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Customer',), ('Invoice',), ('InvoiceLine',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_A6VBFYlY9taV92KtDnSqPKeS)
 Call ID: call_A6VBFYlY9taV92KtDnSqPKeS
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type='table' AND name IN ('Customer','Invoice','InvoiceLine') ORDER BY

In [8]:
print("Memory of the conversation:")
print(steps)

Memory of the conversation:
[{'messages': [HumanMessage(content='This is Frank Harris, What was the total on my last invoice?', additional_kwargs={}, response_metadata={}, id='ad2bd76d-e5f6-4c66-bc00-2167c2ecf8d5')]}, {'messages': [HumanMessage(content='This is Frank Harris, What was the total on my last invoice?', additional_kwargs={}, response_metadata={}, id='ad2bd76d-e5f6-4c66-bc00-2167c2ecf8d5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 521, 'prompt_tokens': 238, 'total_tokens': 759, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYJQD1pV1boMzZEUCByYdKeS66

In [9]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

Could you clarify which titles you mean—titles of what (e.g., books, movies, articles) and from which table or context? If you specify the table and any filters, I can list up to 5 matching titles.


## Add memory

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

In [12]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [13]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_wd95qIEgPuRXQJQ9g6zOxQeN)
 Call ID: call_wd95qIEgPuRXQJQ9g6zOxQeN
  Args:
    query: SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_ET0BgMzGAhvceyZz6BbjRbKH)
 Call ID: call_ET0BgMzGAhvceyZz6BbjRbKH
  Args:
    query: SELECT c.FirstName, c.LastName, i.InvoiceId, i.InvoiceDate, i.Total
FROM Customer AS c
JOIN Invoice AS i ON i.CustomerId = c.CustomerId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
ORDER BY i.InvoiceDate DESC,

In [14]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_FOcFmeEAtMMaNzZCp9sUSkAA)
 Call ID: call_FOcFmeEAtMMaNzZCp9sUSkAA
  Args:
    query: SELECT t.Name AS TrackTitle
FROM Customer AS c
JOIN Invoice AS i ON i.CustomerId = c.CustomerId
JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId
JOIN Track AS t ON t.TrackId = il.TrackId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
  AND i.InvoiceId = (
    SELECT i2.InvoiceId
    FROM Invoice AS i2
    WHERE i2.CustomerId = c.CustomerId
    ORDER BY i2.InvoiceDate DESC, i2.InvoiceId DESC
    LIMIT 1
  )
ORDER BY il.InvoiceLineId
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',)]
================================== A